# D8 flow direction lab
Assign each DEM cell to its steepest downslope neighbour. Explore the effect of cell dimensions, NoData and local sinks.

This lab demonstrates direction assignment only. It does not fill depressions, resolve flats or calculate flow accumulation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/d8-flow'))
from d8_flow import d8_flow_direction


## 1. Inspect a synthetic DEM

The surface slopes generally toward the southeast, with local variation. Each cell's direction is chosen from the eight neighbouring cells by comparing drop per horizontal distance.

In [ ]:
dem = np.array([[120, 118, 116, 114, 112], [119, 117, 115, 113, 111], [118, 116, 114, 112, 110], [117, 115, 113, 111, 109], [116, 114, 112, 110, 108]], dtype=float)
directions = d8_flow_direction(dem, cell_size=(10, 10))
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
im = axes[0].imshow(dem, cmap='terrain'); axes[0].set_title('Elevation'); fig.colorbar(im, ax=axes[0], label='Elevation units')
im2 = axes[1].imshow(dem, cmap='terrain'); axes[1].set_title('D8 directions')
offsets = {1:(1,0), 2:(1,-1), 4:(0,-1), 8:(-1,-1), 16:(-1,0), 32:(-1,1), 64:(0,1), 128:(1,1)}
for r in range(dem.shape[0]):
    for c in range(dem.shape[1]):
        code = int(directions[r, c])
        if code in offsets:
            dx, dy = offsets[code]
            axes[1].quiver(c, r, dx*.35, dy*.35, angles='xy', scale_units='xy', scale=1, color='black')
        else:
            axes[1].text(c, r, 'sink' if code == 0 else 'NoData', ha='center', va='center', fontsize=7)
for ax in axes: ax.set_xticks(range(5)); ax.set_yticks(range(5)); ax.grid(True, alpha=.25)
plt.show()
print(directions)


## 2. Move the hill and change the terrain grid

The next view calls the tested `d8_flow_direction` implementation. Move the synthetic hill, change the east-west tilt and alter row/column spacing. Arrows show the receiving neighbour; an × marks a sink or unresolved flat.

![D8 direction codes around a raster cell.](../assets/d8-direction-codes.svg)


In [ ]:
from ipywidgets import FloatSlider, interact

DIRECTION_OFFSETS = {
    1: (0, 1), 2: (1, 1), 4: (1, 0), 8: (1, -1),
    16: (0, -1), 32: (-1, -1), 64: (-1, 0), 128: (-1, 1),
}

def hill_surface(peak_x=4.0, peak_y=4.0, east_tilt=0.0):
    rows, cols = np.mgrid[0:9, 0:9]
    return -((cols - peak_x) ** 2 + (rows - peak_y) ** 2) + east_tilt * cols

def draw_d8_hill(peak_x=4.0, peak_y=4.0, east_tilt=0.0, cell_width=1.0, cell_height=1.0):
    surface = hill_surface(peak_x, peak_y, east_tilt)
    codes = d8_flow_direction(surface, cell_size=(cell_height, cell_width))
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    image = ax.imshow(surface, origin='upper', cmap='terrain')
    fig.colorbar(image, ax=ax, label='Illustrative elevation')
    for row in range(surface.shape[0]):
        for col in range(surface.shape[1]):
            code = int(codes[row, col])
            if code in DIRECTION_OFFSETS:
                dr, dc = DIRECTION_OFFSETS[code]
                ax.annotate('', xy=(col + 0.38 * dc, row + 0.38 * dr), xytext=(col, row),
                            arrowprops={'arrowstyle': '->', 'linewidth': 1.0})
            elif code == 0:
                ax.scatter(col, row, marker='x', s=24)
    ax.set(title='D8 flow direction on a movable synthetic hill', xlabel='Column', ylabel='Row',
           xticks=range(9), yticks=range(9))
    ax.grid(alpha=0.2)
    plt.show()

interact(
    draw_d8_hill,
    peak_x=FloatSlider(min=2, max=6, step=0.5, value=4, description='Peak column'),
    peak_y=FloatSlider(min=2, max=6, step=0.5, value=4, description='Peak row'),
    east_tilt=FloatSlider(min=-2, max=2, step=0.25, value=0, description='East tilt'),
    cell_width=FloatSlider(min=0.5, max=3, step=0.25, value=1, description='Cell width'),
    cell_height=FloatSlider(min=0.5, max=3, step=0.25, value=1, description='Cell height'),
);


## 3. Change the cell dimensions

A diagonal neighbour is farther away than an axis-aligned neighbour. For rectangular cells, row and column spacing must be treated separately.

In [ ]:
rectangular = d8_flow_direction(dem, cell_size=(10, 25))
print('Square-cell directions:\n', directions)
print('Rectangular-cell directions:\n', rectangular)


## 4. Inspect a pit and a NoData cell

The simple strict-descent rule does not fill depressions or resolve flats. NoData cells are excluded from neighbour selection.

In [ ]:
pit = np.array([[5, 5, 5], [5, 1, 5], [5, 5, 5]], dtype=float)
pit[0, 0] = -9999
pit_directions = d8_flow_direction(pit, nodata=-9999)
print('Pit DEM:\n', pit)
print('D8 codes:\n', pit_directions)


## 5. Investigate

1. Why is the steepest slope not always the neighbour with the largest elevation drop?
2. How does rectangular cell spacing affect diagonal directions?
3. Why do flats and pits remain unresolved here?
4. What would depression filling or flat resolution change?
5. Which CRS, vertical datum, units and NoData policy should be recorded for a real DEM?